- This file contains all the transformations that can be performed outside and before the fols of our cross validation algorithm.
- This correspond to the pre processing functions that will not induce data leackage.
- We decided to perform some of the safe operations outside the folds in order to make this project run more efficiently, by avoiding to reapply this step for all folds, we just do it once. 

In [1]:
import pandas as pd
import unicodedata
import re

Tranformations applied here:
- Year: minimum year and max year;
- Percentages within 0-100%;
- Dropping has damage and carID;
- 

## the upper, strip, erase weird characters, 

This function:
- keeps letters and digit;
- transforms full string to upper case;
- removes spaces in the middle, beggining and end; (might allow spaces in the middle)
- removes symbols and ponctuation; (might allow keeping -)
- removes acentos of the vogals.

In [2]:
def basic_string_transformer(word, 
    remove_middle_spaces: bool = True,
    allow_extra_chars: str = ""
    ):

    if pd.isna(word):
        return word
    
    s = str(word)

    # 1st: strip the leading and trailing whitespaces 
    s = s.strip()

    # 2nd: uppercase the string
    s = s.upper()

    # 3rd: removal of accents
    s = unidecode.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")

    # 4th: remove middle spaces if required
    if remove_middle_spaces:
        s = s.replace(" ", "")
    else:
        s = re.sub(r"\s+", " ", s)
        s = s.strip()
    
    # 5th: remove ponctuation/symbols etc
    # basically, by default, only keeps A-Z characters and digits
    # optionally if the user selects extra allowed characters, we keep them too
    allowed = allowed_extra_chars or "" 
    # if we want to keep middle spaces, we add space to allowed characters
    if not remove_middle_spaces:
        allowed += " "
    # pattern = f"[^A-Z0-9{re.escape(allowed)}]" we did not use this because treats it as a raw string - possible problem with new line 
    pattern = rf"[^A-Z0-9{re.escape(allowed)}]"
    s = re.sub(pattern, "", s)

    #6th: if string becomes empty or only has spaces, return NaN
    if s.strip() == "":
        return np.nan
    
    return s

In [4]:
def def_string_basic_transformer(
    df: pd.DataFrame,
    column: str,
    remove_middle_spaces: bool = True,
    allow_extra_chars: str = ""
    ) -> pd.DataFrame: #returns a dataframe
    """ 
    Apply basic string normalization to a given column of a DataFrame.

    - Parameters: 
    df : pd.DataFrame
        Input DataFrame.
    column : str
        Name of the column to transform.
    remove_inner_spaces : bool, default True
        If True, remove all spaces (including in the middle).
        If False, keep inner spaces (normalizing multiple spaces to just one).
    allowed_extra_chars : str, default ""
        String of extra characters to keep (e.g., "-/" to keep '-' and '/').

    - Returns
    pd.DataFrame
        A copy of df with the specified column transformed.
    """
    
    df_out = df.copy()
    df_out[column] = df_out[column].apply(
        lambda x: basic_string_transformer(
            x,      
            remove_middle_spaces=remove_middle_spaces,
            allow_extra_chars=allow_extra_chars,
        )
    )   
    df_out[column] = df_out[column].astype("string")

    return df_out

# Cross val functions

For the transformations to be performed inside the cross validation folds, we followed a fit and transform logic.
- Fit: the fit functions are the ones that are consist of imputations from the dataset, basically calculations such as median, average, etc. This are calculated based on the training dataset.
- Transform: this consist on the transformations that eiter don't rely on calculations or that apply the previously "fit" imputations made (for example, the actual replacement of missing values with the median). 

in total we have 4 catgeorical features, each of them with their own transformations:
1) brand:
    - correct_invalid_brands_in_df -> transformm 
    - correct_ambiguous_brands -> problem: envolves both fitting and transforming. 2 options:
        - aplly outside fold and say its just a semantic correction;
        - split in 2 fit and transform and store the state
    
2) model
    - correct_invalid_models: same problem with split of fitting and transforming

3) fuelType
    - correct_invalid_fueltypes

4) transmission
    - correct_invalid_transmissions 

## 1) Brand

In [ ]:
# Define the initial list of valid car brands
valid_brands = ['FORD', 'MERCEDES', 'VW', 'OPEL', 'BMW', 'AUDI', 'TOYOTA', 'SKODA', 'HYUNDAI']

# Fill missing Brand values with 'UNKNOWN'
train['Brand'] = train['Brand'].fillna('UNKNOWN') 

# Identify brands in the dataset that are not in the list of valid brands
invalids = sorted(
    [b for b in train['Brand'].unique() if b not in valid_brands],
    key=len
)

# Display the invalid brands found
print("Invalid brands:", invalids)


TRANSFORM THIS TO FUNCTION